<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_10_SBIBM_Hybrid_Benchmark_TwoMoons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 10 — A high-precision hNPE–hNDE benchmark on sbibm

Exercises 5–9 established the hybrid constructions on controlled examples. This exercise is an independent benchmark implementation: none of their algorithms or helper functions is modified.

Version 2 implements the predeclared “winning strategy”:

- a correctly mixed neural spline flow, with learned LU mixing after every coupling transform;
- a small all-data NSF baseline matching the architecture scale used by the published NPE;
- genuine fivefold cross-fitting, so every simulation trains a ratio exactly once and a flow four times;
- one large, four-member posterior correction ensemble in every fold;
- an equally large, four-member hNDE likelihood-ratio ensemble in every fold;
- plateau learning-rate schedules, fresh checkpoints, and diagnostics selected without reference posterior samples;
- full C2ST comparisons for the NPE reference, hNPE, hNDE, and the dual consensus;
- explicit campaigns for both Two Moons and SLCP.

The simulator budget remains exactly 100,000 unique prior-predictive pairs per task. Cross-fitting and ensembling spend computation, not additional simulator calls.


In [ ]:
## ============================================================================
# Google Colab setup — safe to re-run and a no-op off Colab.
# ============================================================================
import os, sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True


def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    # This backward-compatible working directory preserves models from Exercises 4--9.
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)

    # Colab already provides PyTorch, NumPy, SciPy, pandas, and scikit-learn.
    # Installing sbibm without its historical algorithm dependencies avoids
    # downgrading the modern Colab environment; only the task/metric dependencies
    # used below are added explicitly. Version 1.1.0 is the latest official
    # sbibm release and includes the corrected Gaussian Mixture simulator.
    run(sys.executable, "-m", "pip", "install", "-q", "nflows", "pyro-ppl")
    run(
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "sbibm==1.1.0",
    )
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    for candidate in [
        Path.cwd(),
        Path.cwd() / "workshops" / "ml4hep_tifr_colab",
    ]:
        if (candidate / "utils_benchmark.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

print("Working directory:", Path.cwd())


In [ ]:
import gc
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sbibm
import torch
from IPython.display import display

from utils_benchmark import (
    PAPER_ALGORITHMS,
    benchmark_ratio_tail_diagnostics,
    best_published_targets,
    compare_with_paper,
    crossfit_coverage_table,
    draw_hybrid_posterior,
    evaluate_observation_suite,
    evaluate_posterior_samples,
    load_or_simulate_bank,
    load_paper_results,
    posterior_predictive_diagnostics,
    simulation_based_calibration_diagnostics,
    summarize_our_results,
    summarize_paper_results,
    task_recommendation_table,
    train_hybrid_benchmark_model,
    train_official_style_nsf_baseline,
    training_diagnostics,
)
from utils_plotting import export_standalone_figure_script

SEED = 23102026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("sbibm version:", sbibm.__version__)
print("Using device:", device)


## 1. What is in the benchmark?

The paper defines ten tasks from eight simulator families. This campaign focuses on two continuous tasks for which both hybrid paths are well defined:

- **Two Moons** has a curved, multimodal two-dimensional posterior and directly exposes whether a flow mixes both parameter coordinates.
- **SLCP** has five parameters, eight observables, and a four-mode posterior. It is a substantially harder test of both the conditional posterior and the generative hNDE route.

The remaining tasks stay in the recommendation table for later campaigns. The discrete and ODE tasks require additional modeling choices and are not silently forced through a continuous hNDE.


In [ ]:
recommendations = task_recommendation_table()
display(
    recommendations[
        [
            "task", "dim_parameters", "dim_data", "data_type",
            "recommended_method", "colab_status", "reason",
        ]
    ].style.hide(axis="index")
)


## 2. The two hybrid factorizations and the cross-fit

One prior-predictive bank is sampled,

$$
(\theta_i,x_i)\sim\pi(\theta)p(x\mid\theta).
$$

For fold $k$, four fifths train the posterior flow
$q_{\phi,k}(\theta\mid x)$ and the held-out fifth trains

$$
r_{{\rm P},k}(\theta,x)
=
\frac{p(\theta\mid x)}
{(1-\epsilon)q_{\phi,k}(\theta\mid x)+\epsilon\pi(\theta)}.
$$

The independent likelihood route trains a conditional flow and a
residual correction,

$$
q_{\eta,k}(x\mid\theta)\approx p(x\mid\theta),\qquad
c_k(x,\theta)=\frac{p(x\mid\theta)}
{q_{\eta,k}(x\mid\theta)}.
$$

Its normalized likelihood is

$$
\widehat L_k(x;\theta)
=
q_{\eta,k}(x\mid\theta)
\frac{\widehat c_k(x,\theta)}
{\widehat Z_{{\rm C},k}(\theta)}.
$$

Positive and negative classifier rows share either $x_i$ (hNPE) or
$\theta_i$ (hNDE). The simulator indices are split before the class rows
are constructed, so paired rows cannot leak across training and
validation.


## 3. What has to be beaten?

The paper evaluates 10,000 inferred posterior samples against 10,000 reference samples with a z-scored, five-fold neural classifier two-sample test. C2ST $=0.5$ is ideal and lower is better. Results are averaged over ten fixed observations, with simulation budgets of $10^3$, $10^4$, and $10^5$ **per trained inference model**.

The official Gaussian Mixture simulator had a bug that was corrected in `sbibm` v1.1.0. Exercise 10 uses the official rerun table for that task and the manuscript table for the other nine tasks.


In [ ]:
paper_results = load_paper_results(corrected_gaussian_mixture=True)
targets_100k = best_published_targets(paper_results, num_simulations=100_000)
display(
    targets_100k[
        ["task", "algorithm", "mean", "ci95"]
    ].rename(
        columns={
            "algorithm": "best published algorithm",
            "mean": "mean C2ST",
            "ci95": "95% CI",
        }
    ).style.format({"mean C2ST": "{:.4f}", "95% CI": "{:.4f}"}).hide(axis="index")
)


## 4. Choose the campaign

The profiles control computation but never change the simulator accounting. The paper and challenge profiles use five complementary folds and four large ratio networks per fold, for both hNPE and hNDE.

Run the notebook first with Two Moons. Then change only TASK_NAME to SLCP and rerun; the final campaign cell combines the two saved v2 result files.


In [ ]:
PROFILE = "challenge"  # "quick", "paper", or "challenge"
TASK_NAME = "two_moons"
BENCHMARK_TASKS = ("two_moons", "slcp")
LOAD_IF_AVAILABLE = True
RUN_SBC = True
RUN_POSTERIOR_PREDICTIVE = False  # Extra simulator calls.

if TASK_NAME not in BENCHMARK_TASKS:
    raise ValueError(f"TASK_NAME must be one of {BENCHMARK_TASKS}.")

PROFILES = {
    "quick": {
        "num_simulations": 10_000,
        "num_folds": 2,
        "ensemble_size": 2,
        "flow_epochs": 25,
        "baseline_epochs": 25,
        "ratio_epochs": 35,
        "n_proposal": 60_000,
        "observations": [1],
        "sbc_cases_per_fold": 8,
        "sbc_proposal": 256,
        "normalizer_contexts": 512,
        "normalizer_reference": 32,
    },
    "paper": {
        "num_simulations": 100_000,
        "num_folds": 5,
        "ensemble_size": 4,
        "flow_epochs": 70,
        "baseline_epochs": 70,
        "ratio_epochs": 90,
        "n_proposal": 300_000,
        "observations": list(range(1, 11)),
        "sbc_cases_per_fold": 24,
        "sbc_proposal": 512,
        "normalizer_contexts": 2_048,
        "normalizer_reference": 64,
    },
    "challenge": {
        "num_simulations": 100_000,
        "num_folds": 5,
        "ensemble_size": 4,
        "flow_epochs": 90,
        "baseline_epochs": 90,
        "ratio_epochs": 120,
        "n_proposal": 500_000,
        "observations": list(range(1, 11)),
        "sbc_cases_per_fold": 32,
        "sbc_proposal": 768,
        "normalizer_contexts": 2_048,
        "normalizer_reference": 64,
    },
}
campaign = PROFILES[PROFILE]
NUM_SIMULATIONS = campaign["num_simulations"]
N_CROSSFIT_FOLDS = campaign["num_folds"]
RATIO_ENSEMBLE_SIZE = campaign["ensemble_size"]
N_PROPOSAL = campaign["n_proposal"]
OBSERVATIONS = campaign["observations"]
N_POSTERIOR_SAMPLES = 10_000
DEFENSIVE_EPSILON = 0.02
SPLIT_SEED = SEED + 17
TRAIN_HNDE = True
HEADLINE_METHOD = "hNPE"

FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 8,
    "hidden_features": 256,
    "hidden_layers": 3,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": campaign["flow_epochs"],
    "learning_rate": 1.0e-3,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 4,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.1,
    "patience": 18,
    "gradient_clip": 5.0,
}
BASELINE_MODEL_CONFIG = {
    "n_coupling_layers": 5,
    "hidden_features": 50,
    "hidden_layers": 2,
    "spline_num_bins": 10,
    "spline_tail_bound": 3.0,
    "dropout_probability": 0.0,
}
BASELINE_TRAINING_CONFIG = {
    **FLOW_TRAINING_CONFIG,
    "n_epochs": campaign["baseline_epochs"],
    "patience": 15,
}
RATIO_MODEL_CONFIG = {
    "hidden_features": 1024,
    "hidden_layers": 5,
    "dropout_probability": 0.0,
}
RATIO_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": campaign["ratio_epochs"],
    "learning_rate": 1.0e-3,
    "lr_scheduler": "plateau",
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 3,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.15,
    "patience": 25,
    "gradient_clip": 5.0,
}

RUN_TAG = (
    f"{PROFILE}_budget{NUM_SIMULATIONS}_lu8_k{N_CROSSFIT_FOLDS}_"
    f"ratioensemble{RATIO_ENSEMBLE_SIZE}_conditionalref_v3"
)
MODEL_DIR = Path("models_exercise10_sbibm") / TASK_NAME / RUN_TAG
CACHE_DIR = Path("dataframes_exercise10_sbibm")
RESULT_DIR = Path("exercise10_benchmark_results")
FIGURE_SCRIPT_DIR = (
    Path("exercise10_figures_scripts") / TASK_NAME / RUN_TAG
)
for directory in (MODEL_DIR, CACHE_DIR, RESULT_DIR, FIGURE_SCRIPT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "profile": PROFILE,
    "task": TASK_NAME,
    "campaign_tasks": BENCHMARK_TASKS,
    "simulator_budget": NUM_SIMULATIONS,
    "crossfit_folds": N_CROSSFIT_FOLDS,
    "ratio_members_per_fold_per_path": RATIO_ENSEMBLE_SIZE,
    "conditional_reference_hnde": TRAIN_HNDE,
    "normalizer_contexts": campaign["normalizer_contexts"],
    "normalizer_reference_per_context": campaign["normalizer_reference"],
    "headline_method": HEADLINE_METHOD,
    "run_tag": RUN_TAG,
    "observations": OBSERVATIONS,
}, indent=2))


## 5. Acquire the exact-budget simulation bank

The official prior and simulator generate one cached bank per task and budget. The same bank is reused by the all-data NSF baseline, all five hNPE folds, and all five hNDE folds. The coverage check below fails immediately unless every row has exactly one ratio role and four flow roles in the fivefold campaign.


In [ ]:
try:
    task = sbibm.get_task(TASK_NAME)
except (ImportError, ModuleNotFoundError) as exc:
    raise RuntimeError(f"Could not load the official sbibm task {TASK_NAME!r}.") from exc

print(
    f"{task.name_display}: dim(theta)={task.dim_parameters}, "
    f"dim(x)={task.dim_data}, observations={task.num_observations}"
)
simulation_bank = load_or_simulate_bank(
    task,
    NUM_SIMULATIONS,
    cache_path=CACHE_DIR / f"{TASK_NAME}_prior_predictive_{NUM_SIMULATIONS}_seed{SEED}.npz",
    seed=SEED,
    chunk_size=20_000,
)
assert simulation_bank.num_simulations == NUM_SIMULATIONS
print("Unique simulator calls used for every neural object:", simulation_bank.num_simulations)

coverage = crossfit_coverage_table(
    simulation_bank,
    n_folds=N_CROSSFIT_FOLDS,
    split_seed=SPLIT_SEED,
)
display(coverage.style.hide(axis="index"))


## 6. Train the all-data NSF and fivefold dual model

Each fold trains two high-capacity conditional flows:

- $q_{\phi,k}(\theta\mid x)$ for the posterior proposal;
- $q_{\eta,k}(x\mid\theta)$ for the likelihood reference.

The held-out fold trains paired residual classifiers for both routes.
A small regressor estimates $\log Z_{{\rm C},k}(\theta)$ from
conditional-flow samples only, so no simulator calls are added. The v3
checkpoint tag prevents any unconditional-reference v2 model from being
loaded accidentally.


In [ ]:
baseline_model = train_official_style_nsf_baseline(
    task,
    simulation_bank,
    checkpoint=MODEL_DIR / "all_data_official_style_nsf.pt",
    model_config=BASELINE_MODEL_CONFIG,
    training_config=BASELINE_TRAINING_CONFIG,
    device=device,
    seed=SEED + 5_000,
    load_if_available=LOAD_IF_AVAILABLE,
)

hybrid_models = []
for fold in range(N_CROSSFIT_FOLDS):
    print("\n" + "=" * 88)
    print(f"TRAINING CROSSFIT FOLD {fold + 1}/{N_CROSSFIT_FOLDS}")
    print("=" * 88)
    hybrid_models.append(
        train_hybrid_benchmark_model(
            task,
            simulation_bank,
            model_dir=MODEL_DIR,
            flow_model_config=FLOW_MODEL_CONFIG,
            flow_training_config=FLOW_TRAINING_CONFIG,
            ratio_model_config=RATIO_MODEL_CONFIG,
            ratio_training_config=RATIO_TRAINING_CONFIG,
            device=device,
            seed=SEED + 100_000 * fold,
            split_seed=SPLIT_SEED,
            ensemble_size=RATIO_ENSEMBLE_SIZE,
            defensive_epsilon=DEFENSIVE_EPSILON,
            fold=fold,
            n_folds=N_CROSSFIT_FOLDS,
            train_hnde=TRAIN_HNDE,
            load_if_available=LOAD_IF_AVAILABLE,
            retrain_outliers=True,
            normalizer_contexts=campaign["normalizer_contexts"],
            normalizer_reference_per_context=campaign["normalizer_reference"],
        )
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(
    f"Trained the all-data baseline and {len(hybrid_models)} hybrid folds; "
    f"unique simulator calls remain {NUM_SIMULATIONS:,}."
)


In [ ]:
def export_exercise10_figure(fig, script_name):
    return export_standalone_figure_script(
        fig,
        script_name=script_name,
        output_dir=FIGURE_SCRIPT_DIR,
    )


fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
axes[0, 0].plot(
    baseline_model.q_phi.get("history", {}).get("validation", []),
    color="black", ls="--", lw=2, label="all-data NSF",
)
for model in hybrid_models:
    fold_label = f"fold {model.fold}"
    q_phi_history = model.q_phi.get("history", {})
    axes[0, 0].plot(q_phi_history.get("validation", []), label=fold_label)
    axes[1, 1].step(
        np.arange(1, len(q_phi_history.get("learning_rate", [])) + 1),
        q_phi_history.get("learning_rate", []),
        where="post", alpha=0.8,
    )
    for member, pack in enumerate(model.r_p_ensemble):
        history = pack.get("history", {})
        axes[0, 1].plot(
            history.get("validation", []),
            alpha=0.75,
            label=f"{fold_label}, member {member}",
        )
        axes[1, 2].step(
            np.arange(1, len(history.get("learning_rate", [])) + 1),
            history.get("learning_rate", []),
            where="post", alpha=0.55,
        )
    if model.has_hnde:
        q_eta_history = model.q_eta.get("history", {})
        axes[1, 0].plot(q_eta_history.get("validation", []), label=fold_label)
        for member, pack in enumerate(model.r_l_ensemble):
            axes[0, 2].plot(
                pack.get("history", {}).get("validation", []),
                alpha=0.75,
                label=f"{fold_label}, member {member}",
            )

axes[0, 0].set(title=r"conditional $q_\phi(\theta\mid x)$", ylabel="validation NLL")
axes[1, 0].set(title=r"conditional likelihood $q_\eta(x\mid\theta)$", xlabel="epoch", ylabel="validation NLL")
axes[0, 1].set(title=r"posterior correction $r_{\rm P}$", ylabel="validation BCE")
axes[0, 2].set(title=r"conditional residual $c$", ylabel="validation BCE")
axes[1, 1].set(title="flow learning rates", xlabel="epoch", ylabel="learning rate", yscale="log")
axes[1, 2].set(title="ratio learning rates", xlabel="epoch", ylabel="learning rate", yscale="log")
for ax in axes.flat:
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=7, ncol=2)
axes[1, 0].legend(fontsize=7, ncol=2)
export_exercise10_figure(fig, "training_histories_conditionalref_v3")
plt.show()


## 7. Predeclared diagnostics before C2ST

These checks use only the simulation bank and model-generated samples:

- held-out NLL for both conditional flows;
- honest paired-group validation BCE for both residual ensembles;
- a held-out conditional-flow C2ST comparing
  $(\theta,x_{\rm sim})$ with $(\theta,x_{q_\eta})$;
- raw and corrected $Z_{\rm C}(\theta)$ over many held-out parameters;
- member-wise log-ratio tails and arithmetic-ensemble dominance;
- lightweight out-of-fold SBC;
- per-fold and per-observation ESS, maximum weight, and Pareto-$\hat k$.

None of these diagnostics uses the official reference posterior sample.


### Training-bank normalization and SBC checks


In [ ]:
prebenchmark_diagnostics = training_diagnostics(
    hybrid_models,
    simulation_bank,
    max_rows_per_fold=10_000,
    seed=SEED + 600,
)
display(
    prebenchmark_diagnostics.style.format(precision=4).hide(axis="index")
)
assert set(prebenchmark_diagnostics["rP_validation_split"]) == {
    "paired_groups"
}
assert set(prebenchmark_diagnostics["rC_validation_split"]) == {
    "paired_groups"
}

ratio_tail_diagnostics = benchmark_ratio_tail_diagnostics(
    hybrid_models,
    simulation_bank,
    max_rows_per_fold=10_000,
    seed=SEED + 605,
)
display(
    ratio_tail_diagnostics.style.format(precision=4).hide(axis="index")
)

if RUN_SBC:
    sbc_diagnostics = simulation_based_calibration_diagnostics(
        hybrid_models,
        simulation_bank,
        n_cases_per_fold=campaign["sbc_cases_per_fold"],
        n_proposal=campaign["sbc_proposal"],
        seed=SEED + 610,
    )
    display(
        sbc_diagnostics.style.format(precision=4).hide(axis="index")
    )
else:
    sbc_diagnostics = None
    print("SBC diagnostics disabled by configuration.")


### First untouched observation: posterior closure

All routes are now evaluated on the first official observation. Reference posterior samples enter only at this stage, after the architecture, training, diagnostics, and headline method have been fixed.


In [ ]:
NUM_OBSERVATION = 1
observation = task.get_observation(num_observation=NUM_OBSERVATION)
posterior_draws = draw_hybrid_posterior(
    hybrid_models,
    observation.detach().cpu().numpy(),
    baseline_model=baseline_model,
    n_proposal=N_PROPOSAL,
    n_samples=N_POSTERIOR_SAMPLES,
    seed=SEED + 700,
)
display(posterior_draws.diagnostics.style.format(precision=4).hide(axis="index"))

observation1_metrics = evaluate_posterior_samples(
    task,
    NUM_OBSERVATION,
    posterior_draws.samples,
    seed=SEED + 701,
)
observation1_metrics["num_simulations"] = NUM_SIMULATIONS
display(
    observation1_metrics.sort_values("C2ST")
    .style.format({"C2ST": "{:.4f}"})
    .hide(axis="index")
)


In [ ]:
reference_samples = (
    task.get_reference_posterior_samples(num_observation=NUM_OBSERVATION)
    .detach().cpu().numpy()
)
parameter_labels = task.get_labels_parameters()

if task.dim_parameters == 2:
    methods = [
        "all-data NSF baseline",
        "NPE reference",
        "hNPE",
        "hNDE",
        "dual hNPE--hNDE",
    ]
    panels = [("reference posterior", reference_samples)] + [
        (method, posterior_draws.samples[method]) for method in methods
    ]
    n_columns = 3
    n_rows = int(np.ceil(len(panels) / n_columns))
    fig, axes = plt.subplots(
        n_rows, n_columns, figsize=(4.1 * n_columns, 3.8 * n_rows),
        squeeze=False, constrained_layout=True,
    )
    combined = np.concatenate([values for _, values in panels], axis=0)
    x_range = np.quantile(combined[:, 0], [0.002, 0.998])
    y_range = np.quantile(combined[:, 1], [0.002, 0.998])
    for ax, (title, values) in zip(axes.flat, panels):
        ax.hist2d(
            values[:, 0], values[:, 1], bins=65,
            range=[x_range, y_range], cmap="Blues", cmin=1,
        )
        ax.set(title=title, xlabel=parameter_labels[0], ylabel=parameter_labels[1])
    for ax in axes.flat[len(panels):]:
        ax.set_visible(False)
else:
    n_dimensions = min(4, task.dim_parameters)
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
    for dimension, ax in enumerate(axes.flat[:n_dimensions]):
        combined = np.concatenate(
            [
                reference_samples[:, dimension],
                posterior_draws.samples["all-data NSF baseline"][:, dimension],
                posterior_draws.samples["hNPE"][:, dimension],
                posterior_draws.samples["hNDE"][:, dimension],
            ]
        )
        low, high = np.quantile(combined, [0.002, 0.998])
        bins = np.linspace(low, high, 55)
        ax.hist(
            reference_samples[:, dimension], bins=bins, density=True,
            histtype="step", color="black", lw=2, label="reference posterior",
        )
        for method, style in [
            ("all-data NSF baseline", "--"),
            ("NPE reference", ":"),
            ("hNPE", "-"),
            ("hNDE", "-."),
            ("dual hNPE--hNDE", "-"),
        ]:
            ax.hist(
                posterior_draws.samples[method][:, dimension],
                bins=bins, density=True, histtype="step",
                lw=1.5, ls=style, label=method,
            )
        ax.set(xlabel=parameter_labels[dimension], ylabel="posterior density")
        ax.grid(alpha=0.25)
    axes.flat[0].legend(fontsize=7)

export_exercise10_figure(fig, "observation1_posterior_closure_conditionalref_v3")
plt.show()


### Optional posterior-predictive check

A posterior-predictive check needs fresh simulator calls. It is therefore disabled by default and is never folded into the official 100,000-call benchmark result. If enabled, the cell reports the extra call count explicitly and must be treated as a separate diagnostic campaign.


In [ ]:
if RUN_POSTERIOR_PREDICTIVE:
    predictive_diagnostics = posterior_predictive_diagnostics(
        task,
        posterior_draws.samples[HEADLINE_METHOD],
        observation.detach().cpu().numpy(),
        n_simulations=2_000,
        seed=SEED + 800,
    )
    display(predictive_diagnostics.style.format(precision=4).hide(axis="index"))
    print("These 2,000 calls are diagnostic and are outside the official training budget.")
else:
    print(
        "Posterior-predictive simulation is disabled: the headline campaign "
        f"still uses exactly {NUM_SIMULATIONS:,} simulator calls."
    )


## 8. Paper comparison over all ten observations

The paper metric is the mean z-scored, five-fold C2ST over ten fixed
observations. Every route remains visible: the all-data NSF baseline,
cross-fitted NPE reference, hNPE, conditional-reference hNDE, and the
fixed geometric dual consensus.

The result table now retains per-observation ESS, maximum normalized
weight, and Pareto-$\hat k$. A low mean C2ST is not accepted as a healthy
result if it is supported by a collapsed or pathological importance
tail.


In [ ]:
suite_results, _ = evaluate_observation_suite(
    hybrid_models,
    baseline_model=baseline_model,
    observations=OBSERVATIONS,
    n_proposal=N_PROPOSAL,
    n_samples=N_POSTERIOR_SAMPLES,
    seed=SEED + 900,
    keep_draws=False,
)
suite_results["profile"] = PROFILE
suite_results["run_tag"] = RUN_TAG
suite_results["flow_architecture"] = "lu_mixed_nsf_v2"
suite_results["hnde_reference"] = "conditional_q_eta_x_given_theta"

result_path = RESULT_DIR / f"{TASK_NAME}_{RUN_TAG}_c2st.csv"
suite_results.to_csv(result_path, index=False)
print("Saved:", result_path)
display(
    summarize_our_results(suite_results)
    .style.format({"mean": "{:.4f}", "std": "{:.4f}", "ci95": "{:.4f}"})
    .hide(axis="index")
)

weight_health = suite_results.loc[
    suite_results["algorithm"].isin(
        ["hNPE", "hNDE", "dual hNPE--hNDE"]
    ),
    [
        "num_observation",
        "algorithm",
        "ESS",
        "ESS_fraction",
        "max_weight_fraction",
        "pareto_k",
    ],
]
display(
    weight_health.sort_values(["num_observation", "algorithm"])
    .style.format(precision=4)
    .hide(axis="index")
)


In [ ]:
paper_task = summarize_paper_results(
    paper_results,
    num_simulations=NUM_SIMULATIONS,
    task=TASK_NAME,
)
ours_task = summarize_our_results(suite_results)

fig, ax = plt.subplots(figsize=(9.5, 6.0))
paper_order = list(PAPER_ALGORITHMS)
paper_plot = paper_task.set_index("algorithm").reindex(paper_order).dropna().reset_index()
our_plot = ours_task.sort_values("mean")
labels = list(paper_plot["algorithm"]) + list(our_plot["algorithm"])
means = np.concatenate([paper_plot["mean"], our_plot["mean"]])
errors = np.concatenate([paper_plot["ci95"], our_plot["ci95"].fillna(0.0)])
colors = ["0.65"] * len(paper_plot) + [
    "C1" if algorithm == HEADLINE_METHOD else "C0"
    for algorithm in our_plot["algorithm"]
]
y = np.arange(len(labels))
for yi, mean, error, color in zip(y, means, errors, colors):
    ax.errorbar(mean, yi, xerr=error, fmt="o", color=color, capsize=3)
ax.axvline(0.5, color="black", ls="--", lw=1.3, label="ideal C2ST")
ax.set(
    yticks=y,
    yticklabels=labels,
    xlabel="mean C2ST over evaluated observations (lower is better)",
    title=f"{task.name_display}, {NUM_SIMULATIONS:,} simulations",
)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.25)
ax.legend()
export_exercise10_figure(fig, "paper_leaderboard_conditionalref_v3")
plt.show()

comparison = compare_with_paper(
    suite_results,
    paper_results,
    num_simulations=NUM_SIMULATIONS,
    headline_method=HEADLINE_METHOD,
)
display(comparison.style.format(precision=4).hide(axis="index"))

if len(OBSERVATIONS) < 10:
    print(
        "Diagnostic only: the paper averages over ten observations. "
        "Use PROFILE='paper' or 'challenge' for a headline comparison."
    )
elif bool(comparison["beats_published_mean"].iloc[0]):
    print("The predeclared hNPE headline beats the best published mean C2ST.")
else:
    print("The best published mean has not yet been beaten on this task.")


## 9. Two-task campaign: Two Moons and SLCP

Each task is trained and cached separately because the fivefold dual campaign is expensive. Run this notebook once for Two Moons and once for SLCP. The cell below reads only v2 result files and never averages them with the earlier defective-flow run.

SLCP is intentionally subjected to the same large hNPE and hNDE ensembles. Its five-dimensional, four-mode posterior makes it a reach target, but also a much stronger test of whether the dual construction remains useful beyond the two-dimensional diagnostic.


In [ ]:
saved_result_files = sorted(
    RESULT_DIR.glob("*_conditionalref_v3_c2st.csv")
)
if saved_result_files:
    all_our_results = pd.concat(
        [pd.read_csv(path) for path in saved_result_files],
        ignore_index=True,
    )
    all_our_results = all_our_results.loc[
        all_our_results["task"].isin(BENCHMARK_TASKS)
        & (all_our_results["run_tag"] == RUN_TAG)
    ]
    campaign_summary = summarize_our_results(all_our_results)
    display(
        campaign_summary.style.format(
            {"mean": "{:.4f}", "std": "{:.4f}", "ci95": "{:.4f}"}
        ).hide(axis="index")
    )
    completed_tasks = set(all_our_results["task"])
    missing_tasks = [
        name for name in BENCHMARK_TASKS if name not in completed_tasks
    ]
    print(
        "Loaded", len(saved_result_files),
        "conditional-reference v3 campaign file(s)."
    )
    if missing_tasks:
        print("Still to run:", ", ".join(missing_tasks))
    else:
        print(
            "Both Two Moons and SLCP conditional-reference campaigns "
            "are complete."
        )
else:
    print("No conditional-reference v3 result files were found yet.")


## Conclusions and limitations

This v3 notebook replaces the unconditional observation reference by
conditional-reference hNDE:

$$
p(x\mid\theta)
=
q_\eta(x\mid\theta)
\frac{c(x,\theta)}{Z_{\rm C}(\theta)}.
$$

The flow now performs the same kind of density-estimation work as NLE,
while the classifier learns only a residual. The dual construction
keeps its scientific advantages, but $q_\eta(x_o\mid\theta)$ must remain
inside posterior and evidence integrals.

The notebook also fixes the paired train/validation leakage and makes
five diagnostics first-class outputs: conditional-flow C2ST,
parameter-dependent normalization, member-wise ratio tails, per-fold
ESS, and per-observation Pareto-$\hat k$.

The 100,000-simulation comparison remains amortized. It does not add
observation-specific sequential rounds, so SNLE retains an allocation
advantage on SLCP. The purpose of this rerun is to test whether a
classifier-corrected conditional likelihood closes the gap without
changing the simulator budget.
